# Twin Leakage Ledger: LLM attackers on a free Kaggle GPU

Copyright 2026 Paul Burg. Apache-2.0 (code), CC BY 4.0 (this text). Research on synthetic data, no real persons.

Needs no secrets and no paid resources. Before **Run All**, in the right sidebar under *Session options* set:

1. **Accelerator: GPU T4 x2**
2. **Internet: On** (the repository and the model weights are downloaded)
3. Persistence: no persistence is fine

Steps are described in `docs/KAGGLE.md` of the repository.

In [ ]:
# Settings you may change. SET: "reduced" (60 personas, about 3 200 prompts) or "full" (36 000 prompts).
# MODELS: one model per session is safest for the 12 hour limit of a free session.
SET = "reduced"
MODELS = "qwen,mistral,phi"
REF = "prereg-v2"   # tag of the preregistered code; do not change
REPO = "https://github.com/VitaCrypt-Labs/twin-leakage-ledger.git"

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import sys, torch
print(sys.version)
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available(), "gpus:", torch.cuda.device_count())
assert torch.cuda.is_available(), "Switch Accelerator to GPU T4 x2 in Session options"

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone {REPO} repo
%cd /kaggle/working/repo
!git checkout {REF} && git log -1 --format="code commit %H"

In [ ]:
# Pinned extra packages (torch comes from the Kaggle image and is not replaced).
!pip install --no-deps -q -r requirements-llm.txt
!pip list 2>/dev/null | grep -iE "^(transformers|accelerate|huggingface-hub|safetensors|tokenizers|numpy) "

In [ ]:
# Plan and time estimate, nothing is downloaded yet.
!make reproduce-llm SYSTEM_PYTHON=1 LLM_ARGS="--set {SET} --models {MODELS} --dry-run"

In [ ]:
# Optional speed check: 32 answers of the first model (kept in the cache, the later run resumes from them).
FIRST = MODELS.split(",")[0]
!make reproduce-llm SYSTEM_PYTHON=1 LLM_ARGS="--set {SET} --models {FIRST} --limit 32"

In [ ]:
# The real run. Resumable: if the session stops, run this cell again.
!make reproduce-llm SYSTEM_PYTHON=1 LLM_ARGS="--set {SET} --models {MODELS}"

In [ ]:
# Package the answers for download (Output tab, file cache_llm.zip).
!cd /kaggle/working/repo && wc -l cache/llm/*.jsonl && zip -r /kaggle/working/cache_llm.zip cache/llm